In [15]:
from langchain_openai import ChatOpenAI
import os
from pydantic import SecretStr

llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
    temperature=0,
)   
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002E2B2633D70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002E2B2060A70>, root_client=<openai.OpenAI object at 0x000002E2B2341970>

In [16]:
from langchain_community.tools import tool

In [17]:
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def search_duckduckgo(query: str) -> str:
    """Searches DuckDuckGo for the given query to find relevant information and returns the results."""
    duckduckgo_search = DuckDuckGoSearchRun()
    return duckduckgo_search.invoke(query)

#search_duckduckgo.invoke("What is the capital of France?")

In [18]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

@tool
def search_arxiv(query: str) -> str:
    """Searches Arxiv for the given query to find relevant papers and returns the results."""
    arxiv_search = ArxivQueryRun(
        api_wrapper=ArxivAPIWrapper(top_k_results=5, get_full_documents=True)
    )
    return arxiv_search.invoke(query)

#search_arxiv.invoke("quantum computing")

In [19]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def search_wikipedia(query: str) -> str:
    """Searches Wikipedia for the given query to find relevant information and returns the results."""
    wiki_search = WikipediaQueryRun(
        api_wrapper=WikipediaAPIWrapper(top_k_results=5, get_full_documents=True)
    )
    return wiki_search.invoke(query)

#search_wikipedia.invoke("quantum computing")

In [20]:
@tool
def personal_info(name: str):
    """
    Use this tool to get access to get personal information about someone
    Args:
        name (str): _description_
    """
    info={
        "Alice": "Alice is a software engineer with 5 years of experience in web development. She specializes in front-end technologies and has a passion for creating user-friendly interfaces.",
        "Bob": "Bob is a data scientist with a background in machine learning and statistical analysis. He has worked on various projects involving predictive modeling and data visualization.",
        "Charlie": "Charlie is a project manager with expertise in agile methodologies. He has successfully led cross-functional teams and delivered complex projects on time and within budget.",
    }
    return info.get(name, "No information available for this person.")

#personal_info.invoke("Alice")

In [22]:
tools = [search_duckduckgo, search_arxiv, search_wikipedia, personal_info]

llm_with_tools= llm.bind_tools(tools)

In [27]:
response=llm_with_tools.invoke("Who is Alice and what is the capital of France?")
response.tool_calls

[{'name': 'personal_info',
  'args': {'name': 'Alice'},
  'id': 'call_x8VoyzSPvsMPR3zKT3pG70iZ',
  'type': 'tool_call'},
 {'name': 'search_wikipedia',
  'args': {'query': 'Capital of France'},
  'id': 'call_V91AqvpFFQlHFYgXvzPcp5J4',
  'type': 'tool_call'}]